In [1]:
import sys
sys.path.append('/workspaces/BlizzardX')

In [2]:
from src.config.config_manager import ConfigManager
from src.ghcn_daily.ghcn_data_handler import GHCNDataHandler
from src.ghcn_daily.data_fetch import DataFetcher
from src.ghcn_daily.data_processing_old import WeatherDataProcessor
import numpy as np

In [3]:
ghcn = GHCNDataHandler()
config = ConfigManager(config_directory='/workspaces/BlizzardX/src/config')
config.load_config('settings.json')
data_fetcher = DataFetcher(config_file="settings.json",data_type='dataframe')

In [4]:
stations= ghcn.get_station_data(config.get('settings.json', 'data_sources.stations'))
inventory = ghcn.get_inventory_data(config.get('settings.json', 'data_sources.inventory'))

In [5]:
s_state_list = stations[stations['STATE'].isin(['VT', 'NH'])]['ID'].tolist()

s_live_list = inventory[
    (inventory['ID'].isin(s_state_list)) &
    (inventory['LASTYEAR'] >= 2025)  # Station has data this year
]['ID'].unique().tolist()


In [6]:
data= await data_fetcher.save_data(s_live_list)

Fetching Data: 100%|████████████████████████████████████████████████| 33/33 [00:20<00:00,  1.61it/s]


CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is high! Decreasing workers to 2
CPU usage is stable. Increasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6
CPU usage is high! Decreasing workers to 4
CPU usage is stable. Increasing workers to 6


In [7]:
flag_columns = [col for col in data.columns if 'FLAG' in col]
data= data.drop(columns=flag_columns)
#data.replace(-9999.0, np.nan, inplace=True)
weather_variables = ['TMAX', 'TMIN', 'SNOW', 'SNWD', 'PRCP']

In [8]:
from src.ghcn_daily.data_processing import WeatherDataTransformer,WeatherDataImputer
processor = WeatherDataTransformer(data, weather_variables)

In [9]:
df=processor.process_data()

In [10]:
df.replace(-9999.0, np.nan, inplace=True)
df.replace(-999.9, np.nan, inplace=True)

In [11]:
df.head()

,DATE,ID,TMAX,TMIN,SNOW,SNWD,PRCP,Season
0,2009-06-01,US1NHBK0001,NaN,NaN,NaN,NaN,NaN,Summer
1,2009-06-02,US1NHBK0001,NaN,NaN,NaN,NaN,NaN,Summer
2,2009-06-03,US1NHBK0001,NaN,NaN,NaN,NaN,NaN,Summer
3,2009-06-04,US1NHBK0001,NaN,NaN,NaN,NaN,NaN,Summer
4,2009-06-05,US1NHBK0001,NaN,NaN,NaN,NaN,NaN,Summer


In [12]:
import pandas as pd
df = pd.merge(df, stations, on='ID', how='left')
df = df[['DATE','ID', 'LATITUDE', 'LONGITUDE', 'ELEVATION', 'NAME', 'Season', 'TMIN', 'TMAX', 'PRCP', 'SNOW', 'SNWD']]

In [13]:
from src.ghcn_daily.data_filtering import WeatherDataFilter
filter = WeatherDataFilter(df)

In [14]:
df=df[df['ID'].isin(filter.get_stations_with_zero_missing_dates())]

In [15]:
df=df[df['ID'].isin(filter.get_stations_with_low_missing_values(threshold=5))]

In [16]:
imputer= WeatherDataImputer(df)

In [17]:
df=imputer.clean_all()

In [18]:
df.head

<bound method NDFrame.head of               DATE           ID  LATITUDE  LONGITUDE  ELEVATION  \
663411  2008-07-01  USC00272302   42.8267   -71.6261       73.2   
663412  2008-07-02  USC00272302   42.8267   -71.6261       73.2   
663413  2008-07-03  USC00272302   42.8267   -71.6261       73.2   
663414  2008-07-04  USC00272302   42.8267   -71.6261       73.2   
663415  2008-07-05  USC00272302   42.8267   -71.6261       73.2   
...            ...          ...       ...        ...        ...   
1885499 2025-05-15  USW00014755   44.2703   -71.3033     1911.7   
1885500 2025-05-16  USW00014755   44.2703   -71.3033     1911.7   
1885501 2025-05-17  USW00014755   44.2703   -71.3033     1911.7   
1885502 2025-05-18  USW00014755   44.2703   -71.3033     1911.7   
1885503 2025-05-19  USW00014755   44.2703   -71.3033     1911.7   

                     NAME  Season  TMIN  TMAX  PRCP  SNOW  SNWD  
663411       NH E MILFORD  Summer  15.6  28.3   0.0   0.0   0.0  
663412       NH E MILFORD  Summer

In [19]:
df.to_csv('/workspaces/BlizzardX/Data/Predic.csv', index=False)

In [22]:
import pandas as pd
from datetime import datetime, timedelta

# Load data
df = pd.read_csv("/workspaces/BlizzardX/Data/Predic.csv")

# Convert date column
df['DATE'] = pd.to_datetime(df['DATE'])

# Extract state from NAME column
df['STATE'] = df['NAME'].str.extract(r'^([A-Z]{2})')

# Filter for VT and NH
df_filtered = df[df['STATE'].isin(['VT', 'NH'])]

# Define 60-day window
end_date = datetime.today()
start_date = end_date - timedelta(days=60)

# Filter by latest 60 days
df_latest_60 = df_filtered[(df_filtered['DATE'] >= start_date) & (df_filtered['DATE'] <= end_date)]


In [23]:
df_latest_60.head()

,DATE,ID,LATITUDE,LONGITUDE,ELEVATION,NAME,Season,TMIN,TMAX,PRCP,SNOW,SNWD,STATE
6107,2025-03-21,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,3.3,15.0,6.4,0.0,0.0,NH
6108,2025-03-22,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,-3.3,6.1,0.0,0.0,0.0,NH
6109,2025-03-23,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,-4.4,16.7,0.0,0.0,0.0,NH
6110,2025-03-24,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,-3.9,6.1,1.3,0.0,0.0,NH
6111,2025-03-25,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,-2.2,1.7,12.7,0.0,0.0,NH


In [25]:
import joblib
import os

In [27]:
import os
from src.Model.feature_engineering import FeatureEngineering

# Initialize Feature Engineering
fe = FeatureEngineering(df_latest_60)

# Apply all features (rolling averages, lags, cumulative features, interactions)
predictions_features_df = fe.apply_all_features()

print(f"✅Feature Engineered Data Shape: {predictions_features_df.shape}")
print(predictions_features_df.head())

✅Feature Engineered Data Shape: (780, 46)
           DATE   Station_ID  LATITUDE  LONGITUDE  ELEVATION          NAME  \
6107 2025-03-21  USC00272302   42.8267   -71.6261       73.2  NH E MILFORD   
6108 2025-03-22  USC00272302   42.8267   -71.6261       73.2  NH E MILFORD   
6109 2025-03-23  USC00272302   42.8267   -71.6261       73.2  NH E MILFORD   
6110 2025-03-24  USC00272302   42.8267   -71.6261       73.2  NH E MILFORD   
6111 2025-03-25  USC00272302   42.8267   -71.6261       73.2  NH E MILFORD   

      Season  TMIN  TMAX  PRCP  ...  SNWD_TMIN_Interaction  \
6107  Spring   3.3  15.0   6.4  ...                    0.0   
6108  Spring  -3.3   6.1   0.0  ...                   -0.0   
6109  Spring  -4.4  16.7   0.0  ...                   -0.0   
6110  Spring  -3.9   6.1   1.3  ...                   -0.0   
6111  Spring  -2.2   1.7  12.7  ...                   -0.0   

      Snowfall_Intensity SNWD_Snowfall_Diff PRCP_Lag1  PRCP_Lag2  \
6107                 0.0                0.0     

In [29]:
from datetime import datetime, timedelta

# Set date range for the last 14 days
end_date = datetime.today()
start_date = end_date - timedelta(days=14)

# Filter the DataFrame
df_last_2_weeks = predictions_features_df[predictions_features_df['DATE'] >= start_date]

In [30]:
df_last_2_weeks.head(10)

,DATE,Station_ID,LATITUDE,LONGITUDE,ELEVATION,NAME,Season,TMIN,TMAX,PRCP,...,SNWD_TMIN_Interaction,Snowfall_Intensity,SNWD_Snowfall_Diff,PRCP_Lag1,PRCP_Lag2,Cumulative_Precipitation_7,Rolling_Sum_PRCP_14,TMAX_PRCP_Interaction,TMIN_SNOW_Interaction,PRCP_SNOW_Interaction
6153,2025-05-06,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,8.90,12.20,5.6,...,0.0,0.0,0.0,70.1,22.9,106.7,132.0,68.32,0.0,0.0
6154,2025-05-07,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,8.90,13.30,9.9,...,0.0,0.0,0.0,5.6,70.1,116.6,141.9,131.67,0.0,0.0
6155,2025-05-08,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,9.15,18.05,25.0,...,0.0,0.0,0.0,9.9,5.6,141.6,166.9,451.25,0.0,0.0
6156,2025-05-09,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,9.40,22.80,40.1,...,0.0,0.0,0.0,25.0,9.9,176.6,205.5,914.28,0.0,0.0
6157,2025-05-10,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,6.70,9.40,46.5,...,0.0,0.0,0.0,40.1,25.0,220.1,244.4,437.10,0.0,0.0
6158,2025-05-11,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,6.10,16.70,2.8,...,0.0,0.0,0.0,46.5,40.1,200.0,231.5,46.76,0.0,0.0
6159,2025-05-12,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,2.20,18.90,0.0,...,0.0,0.0,0.0,2.8,46.5,129.9,231.0,0.00,0.0,0.0
6160,2025-05-13,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,9.40,25.60,0.0,...,0.0,0.0,0.0,0.0,2.8,124.3,231.0,0.00,0.0,0.0
6161,2025-05-14,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,6.70,22.20,0.0,...,0.0,0.0,0.0,0.0,0.0,114.4,231.0,0.00,0.0,0.0
6162,2025-05-15,USC00272302,42.8267,-71.6261,73.2,NH E MILFORD,Spring,10.00,25.00,2.3,...,0.0,0.0,0.0,0.0,0.0,91.7,233.3,57.50,0.0,0.0


In [42]:
# 1. Prepare feature columns (drop DATE, Station_ID, NAME, TMIN)
non_feature_cols = ['Season', 'TMIN']

# 2. Select feature columns
feature_cols = [col for col in predictions_features_df.columns if col not in non_feature_cols]

# 3. Prepare X for prediction
X_predict_future = df_last_2_weeks[feature_cols].copy()

# 4. Add dummy Cold_Event column if needed
X_predict_future['Cold_Event'] = 0

print(f"✅  X_future ready for prediction: {X_predict_future.shape}")

✅  X_future ready for prediction: (182, 45)


In [38]:
tmin_model = joblib.load('/workspaces/BlizzardX/Notebooks/models/xgboost_tmin_feature_model.pkl')

In [43]:
# Re-add Season column to X_vermont_future from the feature-engineered Vermont dataframe
X_predict_future['Season'] = predictions_features_df['Season']

# Map Season text to numeric codes
season_mapping = {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Fall': 3}
X_predict_future['Season'] = X_predict_future['Season'].map(season_mapping)

print(f"✅ Season column added and mapped correctly! New shape: {X_predict_future.shape}")
print(X_predict_future[['Season']].head())

✅ Season column added and mapped correctly! New shape: (182, 46)
      Season
6153       1
6154       1
6155       1
6156       1
6157       1


In [45]:
X_predict_future.head

<bound method NDFrame.head of              DATE   Station_ID  LATITUDE  LONGITUDE  ELEVATION  \
6153   2025-05-06  USC00272302   42.8267   -71.6261       73.2   
6154   2025-05-07  USC00272302   42.8267   -71.6261       73.2   
6155   2025-05-08  USC00272302   42.8267   -71.6261       73.2   
6156   2025-05-09  USC00272302   42.8267   -71.6261       73.2   
6157   2025-05-10  USC00272302   42.8267   -71.6261       73.2   
...           ...          ...       ...        ...        ...   
103720 2025-05-15  USW00014755   44.2703   -71.3033     1911.7   
103721 2025-05-16  USW00014755   44.2703   -71.3033     1911.7   
103722 2025-05-17  USW00014755   44.2703   -71.3033     1911.7   
103723 2025-05-18  USW00014755   44.2703   -71.3033     1911.7   
103724 2025-05-19  USW00014755   44.2703   -71.3033     1911.7   

                    NAME   TMAX  PRCP  SNOW  SNWD  ... SNWD_Snowfall_Diff  \
6153        NH E MILFORD  12.20   5.6   0.0   0.0  ...                0.0   
6154        NH E MILFOR

In [46]:
# Assuming you have access to station meta info in the same DataFrame
station_id = X_predict_future['Station_ID'].iloc[-1]
station_name = X_predict_future['NAME'].iloc[-1]
state = station_name.split()[0]  # Extracts VT or NH from 'VT XYZ STATION'

# Generate forecast dates
start_date = X_predict_future['DATE'].max() + timedelta(days=1)
forecast_dates = pd.date_range(start=start_date, periods=7)

# Create improved forecast DataFrame
forecast_df = pd.DataFrame({
    'Station_ID': station_id,
    'Station_Name': station_name,
    'State': state,
    'Forecast_Date': forecast_dates,
    'Predicted_TMIN': predictions
})

print(forecast_df)


    Station_ID      Station_Name State Forecast_Date  Predicted_TMIN
0  USW00014755  NH MT WASHINGTON    NH    2025-05-20        1.490417
1  USW00014755  NH MT WASHINGTON    NH    2025-05-21        9.648701
2  USW00014755  NH MT WASHINGTON    NH    2025-05-22       -3.117480
3  USW00014755  NH MT WASHINGTON    NH    2025-05-23       -1.153742
4  USW00014755  NH MT WASHINGTON    NH    2025-05-24      -11.444602
5  USW00014755  NH MT WASHINGTON    NH    2025-05-25       -0.601012
6  USW00014755  NH MT WASHINGTON    NH    2025-05-26        6.443858


In [40]:
last_features = X_predict_future[tmin_model.feature_names_in_].iloc[-1].values.reshape(1, -1)

# Predict next 7 days using rolling forecast
predictions = []
for _ in range(7):
    pred = tmin_model.predict(last_features)[0]
    predictions.append(pred)

    # Shift features and append the new prediction as latest lag
    last_features = np.roll(last_features, -1)
    last_features[0, -1] = pred  # replace the last lag with prediction

# Generate forecast dates
start_date = X_predict_future['DATE'].max() + timedelta(days=1)
forecast_dates = pd.date_range(start=start_date, periods=7)

# Create forecast DataFrame
forecast_df = pd.DataFrame({
    'Forecast_Date': forecast_dates,
    'Predicted_TMIN': predictions
})

print(forecast_df)

  Forecast_Date  Predicted_TMIN
0    2025-05-20        1.490417
1    2025-05-21        9.648701
2    2025-05-22       -3.117480
3    2025-05-23       -1.153742
4    2025-05-24      -11.444602
5    2025-05-25       -0.601012
6    2025-05-26        6.443858
